# 

## 0 · Imports & Base Paths

In [1]:
import pandas as pd
import os

# ── Base directories ──────────────────────────────────────────────────────────
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR     = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/BIOLOGICALPROCESS_MolecularFunction/ALL_BIOLOGICALPROCESS_MolecularFunction.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1 · Load Source Files

Each source DataFrame is loaded, columns lowercased, and `tail_id_is` / `head_id_is`  
set to the appropriate ID namespace (Quick_GO).

### 1.1 Monarch

In [3]:
Monarch_BP_MF = pd.read_csv(PROC_DIR + 'Monarch/Monarch_final/Human/BiologicalProcess_MolecularFunction.csv')
Monarch_BP_MF.columns = Monarch_BP_MF.columns.str.lower()

Monarch_BP_MF['tail_id_is'] = 'Quick_GO'
Monarch_BP_MF['head_id_is'] = 'Quick_GO'
Monarch_BP_MF['relation'] = 'BiologicalProcess_MolecularFunction'
Monarch_BP_MF['head_type'] = 'BiologicalProcess'
Monarch_BP_MF['tail_type'] = 'MolecularFunction'

Monarch_BP_MF['kg_source'] = 'Monarch_KG'
Monarch_BP_MF['kg_type'] = 'Generalised'
Monarch_BP_MF['species'] = 'Homo sapiens'
print(f"Monarch BP→BP: {len(Monarch_BP_MF):,} rows")
Monarch_BP_MF.head(3)

Monarch BP→BP: 834 rows


,head,relation,tail,head_type,relation_type,tail_type,relation_source,head_detail_name,tail_detail_name,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,species_species,species,kg_source,kg_type
0,GO:0060549,BiologicalProcess_MolecularFunction,GO:0042132,BiologicalProcess,related_to,MolecularFunction,infores:go,"regulation of fructose 1,6-bisphosphate 1-phos...","fructose 1,6-bisphosphate 1-phosphatase activity",NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens,Monarch_KG,Generalised
1,GO:0060550,BiologicalProcess_MolecularFunction,GO:0042132,BiologicalProcess,related_to,MolecularFunction,infores:go,"positive regulation of fructose 1,6-bisphospha...","fructose 1,6-bisphosphate 1-phosphatase activity",NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens,Monarch_KG,Generalised
2,GO:0060558,BiologicalProcess_MolecularFunction,GO:0004498,BiologicalProcess,related_to,MolecularFunction,infores:go,regulation of calcidiol 1-monooxygenase activity,calcidiol 1-monooxygenase activity,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens,Monarch_KG,Generalised


## 2 · Consolidate into Unified Schema

All source DataFrames are aligned to `REQUIRED_COLS` (missing columns filled with `pd.NA`),  
then concatenated into `final_df`.

In [4]:
source_dfs = [
    Monarch_BP_MF,
    
]

aligned = []
for df in source_dfs:
    df = df.copy()
    # Drop any duplicated columns that might have resulted from original CSVs
    df = df.loc[:, ~df.columns.duplicated()]
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
final_df = final_df.dropna(subset=['head_detail_name', 'tail_detail_name']).reset_index(drop=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df

Consolidated rows: 834


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,GO:0060549,BiologicalProcess_MolecularFunction,GO:0042132,BiologicalProcess,related_to,MolecularFunction,Monarch_KG,Quick_GO,Quick_GO,"regulation of fructose 1,6-bisphosphate 1-phos...","fructose 1,6-bisphosphate 1-phosphatase activity",Generalised,Homo sapiens
1,GO:0060550,BiologicalProcess_MolecularFunction,GO:0042132,BiologicalProcess,related_to,MolecularFunction,Monarch_KG,Quick_GO,Quick_GO,"positive regulation of fructose 1,6-bisphospha...","fructose 1,6-bisphosphate 1-phosphatase activity",Generalised,Homo sapiens
2,GO:0060558,BiologicalProcess_MolecularFunction,GO:0004498,BiologicalProcess,related_to,MolecularFunction,Monarch_KG,Quick_GO,Quick_GO,regulation of calcidiol 1-monooxygenase activity,calcidiol 1-monooxygenase activity,Generalised,Homo sapiens
3,GO:0060559,BiologicalProcess_MolecularFunction,GO:0004498,BiologicalProcess,related_to,MolecularFunction,Monarch_KG,Quick_GO,Quick_GO,positive regulation of calcidiol 1-monooxygena...,calcidiol 1-monooxygenase activity,Generalised,Homo sapiens
4,GO:0060584,BiologicalProcess_MolecularFunction,GO:0004666,BiologicalProcess,related_to,MolecularFunction,Monarch_KG,Quick_GO,Quick_GO,regulation of prostaglandin-endoperoxide synth...,prostaglandin-endoperoxide synthase activity,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
829,GO:0060314,BiologicalProcess_MolecularFunction,GO:0005219,BiologicalProcess,related_to,MolecularFunction,Monarch_KG,Quick_GO,Quick_GO,regulation of ryanodine-sensitive calcium-rele...,ryanodine-sensitive calcium-release channel ac...,Generalised,Homo sapiens
830,GO:0060315,BiologicalProcess_MolecularFunction,GO:0005219,BiologicalProcess,related_to,MolecularFunction,Monarch_KG,Quick_GO,Quick_GO,negative regulation of ryanodine-sensitive cal...,ryanodine-sensitive calcium-release channel ac...,Generalised,Homo sapiens
831,GO:0060316,BiologicalProcess_MolecularFunction,GO:0005219,BiologicalProcess,related_to,MolecularFunction,Monarch_KG,Quick_GO,Quick_GO,positive regulation of ryanodine-sensitive cal...,ryanodine-sensitive calcium-release channel ac...,Generalised,Homo sapiens
832,GO:0060380,BiologicalProcess_MolecularFunction,GO:0043047,BiologicalProcess,related_to,MolecularFunction,Monarch_KG,Quick_GO,Quick_GO,regulation of single-stranded telomeric DNA bi...,single-stranded telomeric DNA binding,Generalised,Homo sapiens


## 3 · Sanity Check — Distinct Values

In [5]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'BiologicalProcess_MolecularFunction'}
head_type           : {'BiologicalProcess'}
tail_type           : {'MolecularFunction'}
relation_type       : {'related_to'}
kg_source           : {'Monarch_KG'}
head_id_is          : {'Quick_GO'}
tail_id_is          : {'Quick_GO'}


## 4 · NaN Audit (pre-dedup)

In [6]:
true_nan   = final_df.isna().sum()
string_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 5 · Deduplication

Group on `(head, relation, tail)`. For `kg_source`, merge all unique sources with `::` delimiter.  
All other metadata columns take the first non-null value.

In [7]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': 'first',
    'species': 'first'
    
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 834  |  After dedup: 834


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,GO:0000045,BiologicalProcess_MolecularFunction,GO:0019778,BiologicalProcess,related_to,MolecularFunction,Monarch_KG,Quick_GO,Quick_GO,autophagosome assembly,Atg12 activating enzyme activity,Generalised,Homo sapiens
1,GO:0000045,BiologicalProcess_MolecularFunction,GO:0019786,BiologicalProcess,related_to,MolecularFunction,Monarch_KG,Quick_GO,Quick_GO,autophagosome assembly,protein-phosphatidylethanolamide deconjugating...,Generalised,Homo sapiens
2,GO:0000045,BiologicalProcess_MolecularFunction,GO:0061651,BiologicalProcess,related_to,MolecularFunction,Monarch_KG,Quick_GO,Quick_GO,autophagosome assembly,Atg12 conjugating enzyme activity,Generalised,Homo sapiens


## 6 · Post-dedup NaN Audit & Source Distribution

In [8]:
true_nan   = final_df_dedup.isna().sum()
string_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

display(pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
}))

print("\nkg_source values present:", set(final_df_dedup['kg_source']), set(final_df_dedup['kg_type']))

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0



kg_source values present: {'Monarch_KG'} {'Generalised'}


## 7 · Save Output

In [9]:
import os
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 834 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/BIOLOGICALPROCESS_MolecularFunction/ALL_BIOLOGICALPROCESS_MolecularFunction.csv
